In [1]:
!pip install transformer_lens

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 6.1 MB/s eta 0:00:00
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12426 sha256=e99b6f7f35c7476d29198820060d32c7bb00bd467886fe8953f60d937f6ef902
  Stored in directory: /root/.cache/pip/wheels/a8/58/d2/014cb67c3cc6def738c1b1635dbf4e3dab6fb63aba7070dce0
Successfully built transformers-stream-generator


In [ ]:
"""
Phase 2 -- open-ended candidate discovery (SELF-CONTAINED for a fresh
Colab notebook/runtime).

This file merges everything phase2_open_discovery.py needs -- the
foundational classes/functions from phase2_running_total_colab.py
(reused verbatim, not redefined differently) plus the open-discovery-
specific code (PCA-reduced batch Endogeneity screening, budget-limited
Policy-Relevance, all per docs/question_generator_open_discovery_design.md
as corrected). No dependency on any earlier notebook cell -- run this
in a brand-new runtime after `!pip install transformer_lens`.

STATUS: UNTESTED, written outside Colab. Expect debugging. See
phase2_running_total_colab.py's own module docstring for the
simplifications (prompt template, action encoding, provisional
threshold, ad hoc handle_id) that still apply here.
"""

import numpy as np
import torch
from dataclasses import dataclass, field
from enum import Enum
from typing import Dict, List, Optional
from scipy import stats
from sklearn.linear_model import Ridge
from transformer_lens import HookedTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_grad_enabled(False)

model = HookedTransformer.from_pretrained("gpt2", device=DEVICE)
model.eval()
N_LAYERS = model.cfg.n_layers
N_HEADS = model.cfg.n_heads

RNG = np.random.default_rng(0)

def build_lagged(matrix: np.ndarray, lag: int) -> np.ndarray:
    T = matrix.shape[0]
    cols = [matrix[lag - k: T - k] for k in range(1, lag + 1)]
    return np.concatenate(cols, axis=1)



def ols_rss(y: np.ndarray, X: np.ndarray) -> float:
    X_design = np.concatenate([np.ones((X.shape[0], 1)), X], axis=1)
    beta, _, _, _ = np.linalg.lstsq(X_design, y, rcond=None)
    resid = y - X_design @ beta
    return float(resid @ resid)



def benjamini_hochberg(p_values: list, alpha: float = ALPHA) -> list:
    m = len(p_values)
    order = np.argsort(p_values)
    sorted_p = np.array(p_values)[order]
    thresholds = (np.arange(1, m + 1) / m) * alpha
    passed = sorted_p <= thresholds
    if not passed.any():
        cutoff_rank = -1
    else:
        cutoff_rank = np.max(np.where(passed))
    reject = np.zeros(m, dtype=bool)
    if cutoff_rank >= 0:
        reject[order[: cutoff_rank + 1]] = True
    return reject.tolist()


# =============================================================================
# 2. PORTED, VERBATIM: HypothesisEngine
#    (copied from core/hypothesis/hypothesis_engine.py)
# =============================================================================


class NodeStatus(Enum):
    CANDIDATE = "candidate"
    ACCEPTED = "accepted"
    REJECTED = "rejected"
    UNRESOLVED = "unresolved"


@dataclass
class Node:
    signal_id: str
    status: NodeStatus
    endogeneity_p_value: Optional[float] = None
    best_handle_id: Optional[str] = None
    flip_rate: Optional[float] = None
    entity_id: Optional[int] = None
    evidence: dict = field(default_factory=dict)


class WorldModel:
    def __init__(self):
        self.nodes: Dict[str, Node] = {}

    def add_candidate(self, signal_id: str, p_value: float):
        if signal_id not in self.nodes:
            self.nodes[signal_id] = Node(signal_id=signal_id, status=NodeStatus.CANDIDATE,
                                          endogeneity_p_value=p_value)

    def accept(self, signal_id: str, handle_id: str, flip_rate: float, evidence: Optional[dict] = None):
        node = self.nodes[signal_id]
        node.status = NodeStatus.ACCEPTED
        node.best_handle_id = handle_id
        node.flip_rate = flip_rate
        if evidence:
            node.evidence.update(evidence)

    def reject(self, signal_id: str, handle_id: Optional[str] = None, flip_rate: Optional[float] = None):
        node = self.nodes[signal_id]
        node.status = NodeStatus.REJECTED
        if handle_id:
            node.best_handle_id = handle_id
            node.flip_rate = flip_rate

    def self_model_summary(self) -> str:
        lines = []
        by_status: Dict[NodeStatus, List[str]] = {}
        for sid, node in self.nodes.items():
            by_status.setdefault(node.status, []).append(sid)
        for status in NodeStatus:
            ids = by_status.get(status, [])
            if ids:
                lines.append(f"  {status.value:<12}: {len(ids)} signal(s) -- {ids}")
        return "\n".join(lines) if lines else "  (empty -- no evidence accumulated yet)"


# =============================================================================
# 4. PORTED subset: NodeKind, SelfGraphNode
#    (copied from core/self_model/self_graph.py -- Kind axis only, the
#    full edge-typed SelfGraph isn't needed for this validation run)
# =============================================================================


class NodeKind(Enum):
    OBSERVATION = "observation"
    MECHANISM = "mechanism"
    HANDLE = "handle"
    ATOMIC_SIGNAL = "atomic_signal"
    COMPOSITE_SIGNAL = "composite_signal"
    DECOY = "decoy"
    UNKNOWN = "unknown"


@dataclass
class SelfGraphNode:
    node_id: str
    kind: NodeKind = NodeKind.UNKNOWN
    kind_confidence: float = 0.0
    evidence: dict = field(default_factory=dict)

    # Fourth axis (docs/knowledge_unit_reconciliation.md section 4), added after
    # Phase 2's baseline-accuracy check found GPT-2's greedy "a + b ="
    # output matches operand `b` in 11/16 cases -- an induction/copy
    # pattern, not addition. kind/kind_confidence/evidence are the
    # STRUCTURAL finding (Endogenous? Policy-Relevant?) and are
    # unaffected by this; semantic_label is a SEPARATE, falsifiable
    # guess about what the mechanism DOES, kept "UNKNOWN" until an
    # actual discriminating test is run (see section 4 for what that test
    # would need to check -- token-repetition distance vs. numeric
    # value, not just "looks like a match").
    semantic_label: str = "UNKNOWN"
    semantic_confidence: float = 0.0
    semantic_evidence: dict = field(default_factory=dict)


# =============================================================================
# 5. NEW: KnowledgeUnitView
#    (docs/knowledge_unit_reconciliation.md, Option B -- thin, stateless,
#    read-only wrapper. Holds no state of its own.)
# =============================================================================


class KnowledgeUnitView:
    def __init__(self, kind_node: SelfGraphNode, world_model: WorldModel):
        self._kind_node = kind_node
        self._wm = world_model

    @property
    def node_id(self) -> str:
        return self._kind_node.node_id

    @property
    def kind(self) -> NodeKind:
        return self._kind_node.kind

    @property
    def kind_confidence(self) -> float:
        return self._kind_node.kind_confidence

    @property
    def status(self) -> Optional[NodeStatus]:
        node = self._wm.nodes.get(self.node_id)
        return node.status if node else None

    @property
    def endogeneity_p_value(self) -> Optional[float]:
        node = self._wm.nodes.get(self.node_id)
        return node.endogeneity_p_value if node else None

    @property
    def flip_rate(self) -> Optional[float]:
        node = self._wm.nodes.get(self.node_id)
        return node.flip_rate if node else None

    def __repr__(self):
        return (f"KnowledgeUnitView({self.node_id!r}, kind={self.kind.value}, "
                f"kind_confidence={self.kind_confidence:.2f}, "
                f"status={self.status.value if self.status else None}, "
                f"endogeneity_p={self.endogeneity_p_value}, "
                f"flip_rate={self.flip_rate})")


# =============================================================================
# 6. NEW: EvidenceManager
#    (docs/knowledge_unit_reconciliation.md, Option A+C -- hard-required
#    intervention arg, two separate gates)
# =============================================================================

POLICY_RELEVANCE_FLIP_THRESHOLD = 0.3  # PROVISIONAL, see module docstring section 3



class EvidenceManager:
    def __init__(self, world_model: WorldModel, self_graph_nodes: Dict[str, SelfGraphNode]):
        self.wm = world_model
        self.sg_nodes = self_graph_nodes

    def _ensure_node(self, node_id: str) -> SelfGraphNode:
        if node_id not in self.sg_nodes:
            self.sg_nodes[node_id] = SelfGraphNode(node_id=node_id)
        return self.sg_nodes[node_id]

    def record_endogeneity(self, node_id: str, p_value: float, significant: bool) -> None:
        """Correlational gate only. If significant, node becomes a
        WorldModel CANDIDATE. If not, no WorldModel node is created --
        Question Generator has nothing further to ask (section 3.2 of the
        design doc), kind stays UNKNOWN (we asked one question, got a
        no, and did not investigate further -- NOT the same as
        confidently classifying it DECOY, which would need its own
        evidence)."""
        node = self._ensure_node(node_id)
        node.evidence["endogeneity_p_value"] = p_value
        node.evidence["endogeneity_significant"] = significant
        if significant:
            self.wm.add_candidate(node_id, p_value)

    def record_intervention(self, node_id: str, handle_id: str, flip_rate: float,
                             evidence: Optional[dict] = None) -> None:
        """intervention_evidence (handle_id, flip_rate) is NOT optional
        -- there is no code path to ACCEPTED that skips this method
        with real values. Threshold is provisional, see module docstring."""
        node = self._ensure_node(node_id)
        if flip_rate >= POLICY_RELEVANCE_FLIP_THRESHOLD:
            self.wm.accept(node_id, handle_id, flip_rate, evidence)
            node.kind = NodeKind.MECHANISM
            node.kind_confidence = min(1.0, flip_rate)
        else:
            self.wm.reject(node_id, handle_id, flip_rate)
            node.kind = NodeKind.DECOY
            node.kind_confidence = 0.5


# =============================================================================
# 7. NEW: QuestionGenerator
#    (docs/knowledge_unit_reconciliation.md Option B -- independent
#    component, NOT a Hypothesis Generator replacement. Narrow scope
#    here: exactly the two example questions from the reconciliation
#    note, exercised on hand-picked candidates only -- see design doc section 5
#    for why open-ended use is out of scope for Phase 2.)
# =============================================================================


class QuestionGenerator:
    def next_question(self, kuv: KnowledgeUnitView) -> Optional[str]:
        if kuv.status is None:
            return "is_internal"          # -> Endogeneity test
        if kuv.status == NodeStatus.CANDIDATE:
            return "is_controllable"      # -> Policy-Relevance test
        return None                        # ACCEPTED/REJECTED: nothing further


# =============================================================================
# 8. NEW: RunningTotalEnvironment
#    Simplification #1 (see module docstring): uses Phase 1's VALIDATED
#    "{T} + {n} =" format, looped, not the design doc's illustrative
#    "Total: {T}. Add {n}. New total:" phrasing.
# =============================================================================


class RunningTotalEnvironment:
    # [FIXED after first Colab run] start_total no longer defaults to 0.
    # Diagnosed via phase2_diagnose_granger.py: starting at T=0 produced
    # an absorbing fixed point (all 100 steps stuck at T=0), because the
    # first prompt "0 + n =" was outside Phase 1's validated [1,4] range
    # and GPT-2's greedy completion for it happened to be "0" regardless
    # of n. start_range draws the INITIAL total from the validated range
    # instead; later, organically-reached values (including 0) are left
    # untouched -- forcing them would mean testing a constructed signal,
    # not the model's real behavior. A stuck-trajectory watchdog is added
    # so any future absorbing state is caught immediately, not silently
    # consumed over a full run.
    N_STUCK_WARN = 8

    def __init__(self, model, operand_range=(1, 4), start_range=(1, 4), seed=0):
        self.model = model
        self.operand_range = operand_range
        self.start_range = start_range
        self.rng = np.random.default_rng(seed)
        self.T = None
        self.n_anomalies = 0
        self.n_steps_run = 0
        self._stuck_count = 0
        self._last_T = None
        self._warned_stuck = False

    def reset(self):
        self.T = int(self.rng.integers(self.start_range[0], self.start_range[1] + 1))
        self.n_anomalies = 0
        self.n_steps_run = 0
        self._stuck_count = 0
        self._last_T = None
        self._warned_stuck = False

    def step(self) -> dict:
        n = int(self.rng.integers(self.operand_range[0], self.operand_range[1] + 1))
        prompt = f"{self.T} + {n} ="
        tokens = self.model.to_tokens(prompt)
        logits = self.model(tokens)
        next_token_id = int(logits[0, -1, :].argmax().item())
        next_token_str = self.model.to_string(torch.tensor([next_token_id]))

        try:
            T_next = int(next_token_str.strip())
            valid = 0 <= T_next <= 9
        except ValueError:
            valid = False
            T_next = None

        if not valid:
            self.n_anomalies += 1
            # Fall back to wrapping the TRUE sum mod 10 so the loop can
            # keep running -- this step's action is still whatever the
            # model generated (for the action encoding / endogeneity
            # test below), only the CONTINUATION uses the fallback.
            T_next_for_continuation = (self.T + n) % 10
        else:
            T_next_for_continuation = T_next

        # stuck-trajectory watchdog -- catch an absorbing state
        # immediately instead of silently completing a degenerate run
        if self._last_T is not None and T_next_for_continuation == self._last_T:
            self._stuck_count += 1
        else:
            self._stuck_count = 0
        self._last_T = T_next_for_continuation
        if self._stuck_count >= self.N_STUCK_WARN and not self._warned_stuck:
            print(f"  !! WARNING: T stuck at {T_next_for_continuation} for "
                  f"{self._stuck_count + 1} consecutive steps -- possible "
                  f"absorbing state, same failure mode as the T=0 bug")
            self._warned_stuck = True

        result = {
            "T": self.T, "n": n, "prompt": prompt,
            "generated_token_id": next_token_id,
            "generated_token_str": next_token_str,
            "T_next_raw": T_next,          # None if unparseable
            "T_next_valid": valid,
            "T_next": T_next_for_continuation,  # always 0-9, used to continue the loop
        }
        self.T = T_next_for_continuation
        self.n_steps_run += 1
        return result



def action_onehot_from_digit(digit: int, n_categories: int = 10) -> list:
    """Simplification #2 (see module docstring): plain one-hot, not the
    k-1 reduced encoding evaluation/action_encoding.py uses."""
    row = [0.0] * n_categories
    row[digit] = 1.0
    return row


# =============================================================================
# 9. NEW: candidate signal extraction -- fixed, out-of-sample-calibrated
#    probe directions (calibrated ONCE, before any trajectory, on an
#    independent set, per Phase 1's corrected probing methodology)
# =============================================================================


def calibrated_addition_pairs(operand_max=9, seed=7):
    """Same balanced-enumeration, single-token-answer-filtered approach
    Phase 1's probing fix used -- avoids the CV-leakage failure mode
    documented in docs/llm_phase1_addition_circuit.md."""
    rng = np.random.default_rng(seed)
    candidates = [(a, b) for a in range(0, 10) for b in range(0, 10) if a + b <= operand_max]
    valid = []
    for a, b in candidates:
        ans_ids = model.to_tokens(f" {a + b}", prepend_bos=False)
        if ans_ids.shape[-1] == 1:
            valid.append((a, b))
    rng.shuffle(valid)
    return valid



def policy_relevance_flip_rate(env: RunningTotalEnvironment, layers: List[int],
                                n_trials: int = 30, position: int = -1, seed: int = 1) -> float:
    """For n_trials random steps: run a 'clean' prompt, a 'donor' prompt
    (same T, different n), patch the donor's activation at `layers`
    (resid_post, given position) into the clean run, and check whether
    the model's generated next-token ACTION changes. flip_rate = fraction
    of trials where it does."""
    rng = np.random.default_rng(seed)
    flips = 0
    for _ in range(n_trials):
        T = int(rng.integers(0, 10))
        n_clean = int(rng.integers(env.operand_range[0], env.operand_range[1] + 1))
        n_donor = int(rng.integers(env.operand_range[0], env.operand_range[1] + 1))
        while n_donor == n_clean:
            n_donor = int(rng.integers(env.operand_range[0], env.operand_range[1] + 1))

        clean_prompt = f"{T} + {n_clean} ="
        donor_prompt = f"{T} + {n_donor} ="

        clean_tokens = model.to_tokens(clean_prompt)
        donor_tokens = model.to_tokens(donor_prompt)
        if clean_tokens.shape[-1] != donor_tokens.shape[-1]:
            continue  # alignment guard, same discipline as Phase 1

        clean_logits_orig = model(clean_tokens)
        original_action = int(clean_logits_orig[0, -1, :].argmax().item())

        _, donor_cache = model.run_with_cache(donor_tokens)

        def patch_hook(activation, hook, donor_cache=donor_cache, position=position):
            activation[:, position, :] = donor_cache[hook.name][:, position, :]
            return activation

        hooks = [(f"blocks.{layer}.hook_resid_post", patch_hook) for layer in layers]
        patched_logits = model.run_with_hooks(clean_tokens, fwd_hooks=hooks)
        patched_action = int(patched_logits[0, -1, :].argmax().item())

        if patched_action != original_action:
            flips += 1
    return flips / n_trials if n_trials > 0 else 0.0



def policy_relevance_flip_rate_head(env: RunningTotalEnvironment, layer: int, head: int,
                                     n_trials: int = 30, position: int = -1, seed: int = 1) -> float:
    """Same as above but patches a single attention head's OUTPUT
    (hook_z, this head's slice) rather than the whole layer's resid_post."""
    rng = np.random.default_rng(seed)
    flips = 0
    for _ in range(n_trials):
        T = int(rng.integers(0, 10))
        n_clean = int(rng.integers(env.operand_range[0], env.operand_range[1] + 1))
        n_donor = int(rng.integers(env.operand_range[0], env.operand_range[1] + 1))
        while n_donor == n_clean:
            n_donor = int(rng.integers(env.operand_range[0], env.operand_range[1] + 1))

        clean_prompt = f"{T} + {n_clean} ="
        donor_prompt = f"{T} + {n_donor} ="
        clean_tokens = model.to_tokens(clean_prompt)
        donor_tokens = model.to_tokens(donor_prompt)
        if clean_tokens.shape[-1] != donor_tokens.shape[-1]:
            continue

        clean_logits_orig = model(clean_tokens)
        original_action = int(clean_logits_orig[0, -1, :].argmax().item())

        _, donor_cache = model.run_with_cache(donor_tokens)

        def patch_head_hook(activation, hook, donor_cache=donor_cache, position=position, head=head):
            activation[:, position, head, :] = donor_cache[hook.name][:, position, head, :]
            return activation

        hook_name = f"blocks.{layer}.attn.hook_z"
        patched_logits = model.run_with_hooks(clean_tokens, fwd_hooks=[(hook_name, patch_head_hook)])
        patched_action = int(patched_logits[0, -1, :].argmax().item())

        if patched_action != original_action:
            flips += 1
    return flips / n_trials if n_trials > 0 else 0.0




# =============================================================================
# OPEN-DISCOVERY-SPECIFIC CODE (docs/question_generator_open_discovery_design.md)
# =============================================================================

N_STEPS = 200          # bigger than the 100 used for 2-3 candidates --
                        # see design doc's cost note, even with PCA controls
LAG = 3
K_PCA = 5               # top-K principal components used as controls
ALPHA = 0.05
BUDGET = 10              # Stage 2 candidates per round, priority-ordered
N_FLIP_TRIALS = 20       # reduced from 30 to control total runtime at scale

# =============================================================================
# 1. Candidate space: 12 layers + 144 heads
# =============================================================================

LAYER_IDS = [f"layer{l}" for l in range(N_LAYERS)]
HEAD_IDS = [f"head{l}_{h}" for l in range(N_LAYERS) for h in range(N_HEADS)]
CANDIDATE_IDS = LAYER_IDS + HEAD_IDS
print(f"[setup] {len(CANDIDATE_IDS)} candidates "
      f"({len(LAYER_IDS)} layers + {len(HEAD_IDS)} heads)")


def calibrate_all_directions(calibration_pairs):
    """Same technique as Phase 2's hand-picked candidates (Ridge probe
    fit on an independent calibration set, out-of-sample per Phase 1's
    corrected methodology), applied uniformly to all 156 locations."""
    layer_acts = {l: [] for l in range(N_LAYERS)}
    head_acts = {(l, h): [] for l in range(N_LAYERS) for h in range(N_HEADS)}
    sums = []
    for a, b in calibration_pairs:
        prompt = f"{a} + {b} ="
        tokens = model.to_tokens(prompt)
        _, cache = model.run_with_cache(tokens)
        for l in range(N_LAYERS):
            layer_acts[l].append(cache["resid_post", l][0, -1, :].cpu().numpy())
            z = cache["z", l][0, -1, :, :].cpu().numpy()  # [n_heads, d_head]
            for h in range(N_HEADS):
                head_acts[(l, h)].append(z[h])
        sums.append(a + b)
    sums = np.array(sums)

    layer_dirs = {}
    for l in range(N_LAYERS):
        X = np.stack(layer_acts[l])
        ridge = Ridge(alpha=10.0)
        ridge.fit(X, sums)
        w = ridge.coef_
        layer_dirs[l] = w / (np.linalg.norm(w) + 1e-8)

    head_dirs = {}
    for key, acts in head_acts.items():
        X = np.stack(acts)
        ridge = Ridge(alpha=10.0)
        ridge.fit(X, sums)
        w = ridge.coef_
        head_dirs[key] = w / (np.linalg.norm(w) + 1e-8)

    return layer_dirs, head_dirs


print("[setup] calibrating 156 probe directions (one forward pass per "
      "calibration pair, all layers/heads read from the same cache)...")
layer_dirs, head_dirs = calibrate_all_directions(calibrated_addition_pairs())


# =============================================================================
# 2. Trajectory collection: ONE run_with_cache per step, all 156 signals
#    read from that same cache (not 156 separate forward passes per step).
# =============================================================================

def collect_trajectory(env, n_steps):
    rows = []
    actions = []
    env.reset()
    for _ in range(n_steps):
        step_result = env.step()
        tokens = model.to_tokens(step_result["prompt"])
        _, cache = model.run_with_cache(tokens)
        row = {}
        for l in range(N_LAYERS):
            act = cache["resid_post", l][0, -1, :].cpu().numpy()
            row[f"layer{l}"] = float(np.dot(act, layer_dirs[l]))
        for l in range(N_LAYERS):
            z = cache["z", l][0, -1, :, :].cpu().numpy()
            for h in range(N_HEADS):
                row[f"head{l}_{h}"] = float(np.dot(z[h], head_dirs[(l, h)]))
        rows.append(row)
        actions.append(action_onehot_from_digit(step_result["T_next"]))
    matrix = np.array([[row[cid] for cid in CANDIDATE_IDS] for row in rows])
    action_onehot = np.array(actions)
    return matrix, action_onehot, env.n_anomalies


# =============================================================================
# 3. PCA-reduced conditional Granger test (the corrected version --
#    see docs/question_generator_open_discovery_design.md section 2)
# =============================================================================

def conditional_granger_f_test_pca(target_col, other_matrix, action_onehot,
                                    lag=LAG, k_pca=K_PCA):
    T = target_col.shape[0]
    y = target_col[lag:]
    n = y.shape[0]

    self_lags = build_lagged(target_col.reshape(-1, 1), lag)

    if other_matrix.shape[1] > 0:
        k_eff = min(k_pca, other_matrix.shape[1])
        mean = other_matrix.mean(axis=0)
        centered = other_matrix - mean
        _, _, Vt = np.linalg.svd(centered, full_matrices=False)
        components = centered @ Vt[:k_eff].T
        other_lags = build_lagged(components, lag)
    else:
        other_lags = np.zeros((n, 0))

    action_contemporaneous = action_onehot[lag:]
    action_past_lags = build_lagged(action_onehot, lag)
    action_history = np.concatenate([action_contemporaneous, action_past_lags], axis=1)

    restricted_X = np.concatenate([self_lags, other_lags], axis=1)
    unrestricted_X = np.concatenate([restricted_X, action_history], axis=1)

    k_r = restricted_X.shape[1] + 1
    k_u = unrestricted_X.shape[1] + 1
    q = k_u - k_r
    df2 = n - k_u

    rss_r = ols_rss(y, restricted_X)
    rss_u = ols_rss(y, unrestricted_X)

    if df2 <= 0 or rss_u <= 0:
        return float("nan"), 1.0, df2

    f_stat = max(((rss_r - rss_u) / q) / (rss_u / df2), 0.0)
    p_value = stats.f.sf(f_stat, q, df2)
    return f_stat, p_value, df2


def test_endogeneity_batch(matrix, action_onehot, candidate_ids, lag=LAG,
                            k_pca=K_PCA, alpha=ALPHA):
    p_values = []
    df2s = []
    for i in range(matrix.shape[1]):
        target = matrix[:, i]
        others = np.delete(matrix, i, axis=1)
        _, p, df2 = conditional_granger_f_test_pca(target, others, action_onehot,
                                                     lag=lag, k_pca=k_pca)
        p_values.append(p)
        df2s.append(df2)
    rejected = benjamini_hochberg(p_values, alpha=alpha)
    print(f"[Stage 1] df2 across {len(candidate_ids)} tests: "
          f"min={min(df2s)}, mean={np.mean(df2s):.1f} "
          f"(must be > 0 and comfortably large for the F-test to be valid)")
    return {
        candidate_ids[i]: {"p_value": p_values[i], "significant": rejected[i], "df2": df2s[i]}
        for i in range(len(candidate_ids))
    }


# =============================================================================
# 4. Stage 2: budget-limited, priority-ordered Policy-Relevance
# =============================================================================

def flip_rate_for_candidate(env, candidate_id, n_trials=N_FLIP_TRIALS, seed=1):
    if candidate_id.startswith("head"):
        rest = candidate_id[len("head"):]
        l_str, h_str = rest.split("_")
        return policy_relevance_flip_rate_head(env, layer=int(l_str), head=int(h_str),
                                                n_trials=n_trials, seed=seed)
    else:
        l = int(candidate_id[len("layer"):])
        return policy_relevance_flip_rate(env, [l], n_trials=n_trials, seed=seed)


def plan_round(endog_results, budget=BUDGET):
    """QuestionGenerator.plan_round, per docs/question_generator_open_discovery_design.md
    section 4 -- candidates that passed Stage 1, sorted by ascending p-value,
    truncated to `budget`."""
    significant = [(cid, r["p_value"]) for cid, r in endog_results.items() if r["significant"]]
    significant.sort(key=lambda x: x[1])
    return [cid for cid, _ in significant[:budget]]


# =============================================================================
# 5. Main: collect trajectory, batch-screen, budget-test, route through
#    KnowledgeUnitView / EvidenceManager
# =============================================================================

env = RunningTotalEnvironment(model, seed=0)
print(f"\n[collection] gathering {N_STEPS}-step trajectory for 156 candidates...")
matrix, action_onehot, n_anomalies = collect_trajectory(env, N_STEPS)
print(f"[collection] done. generation anomalies: {n_anomalies}/{N_STEPS}")

print("\n[Stage 1] batch Endogeneity screening (PCA-reduced controls)...")
endog_results = test_endogeneity_batch(matrix, action_onehot, CANDIDATE_IDS)

n_significant = sum(r["significant"] for r in endog_results.values())
print(f"[Stage 1] {n_significant}/{len(CANDIDATE_IDS)} candidates passed "
      f"Endogeneity (BH-FDR corrected, alpha={ALPHA})")

world_model = WorldModel()
self_graph_nodes = {}
evidence_mgr = EvidenceManager(world_model, self_graph_nodes)

for cid, r in endog_results.items():
    evidence_mgr.record_endogeneity(cid, r["p_value"], r["significant"])

round_candidates = plan_round(endog_results, budget=BUDGET)
print(f"\n[Stage 2] testing top {len(round_candidates)} candidates by "
      f"ascending p-value (budget={BUDGET}):")
for cid in round_candidates:
    print(f"  {cid}: p={endog_results[cid]['p_value']:.4g}")

print(f"\n[Stage 2] running Policy-Relevance flip-rate tests "
      f"({N_FLIP_TRIALS} trials each)...")
for cid in round_candidates:
    flip_rate = flip_rate_for_candidate(env, cid)
    handle_id = f"donor_activation@{cid}"
    evidence_mgr.record_intervention(cid, handle_id, flip_rate)
    kuv = KnowledgeUnitView(self_graph_nodes[cid], world_model)
    print(f"  {cid}: flip_rate={flip_rate:.3f} -> {kuv}")

print("\n" + "=" * 60)
print("WorldModel summary (open discovery round):")
print(world_model.self_model_summary())

accepted = [cid for cid, node in world_model.nodes.items()
            if node.status == NodeStatus.ACCEPTED]
print(f"\nAccepted candidates this round: {accepted}")
if "layer9" in accepted or "layer10" in accepted or "layer11" in accepted or "head9_1" in accepted:
    print("(sanity check: at least one of the Phase 2-validated layer-9-11 "
          "region candidates reappeared here -- consistent with the earlier "
          "hand-picked validation, good sign the open search isn't "
          "contradicting it)")

np.savez(
    "phase2_open_discovery_results.npz",
    candidate_ids=np.array(CANDIDATE_IDS),
    p_values=np.array([endog_results[c]["p_value"] for c in CANDIDATE_IDS]),
    significant=np.array([endog_results[c]["significant"] for c in CANDIDATE_IDS]),
    tested_stage2=np.array(round_candidates),
    n_anomalies=n_anomalies,
)
print("\nSaved phase2_open_discovery_results.npz")

/tmp/ipykernel_1575/2829871845.py:31: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model = HookedTransformer.from_pretrained("gpt2", device=DEVICE)


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Loaded pretrained model gpt2 into HookedTransformer


NameError: name 'ALPHA' is not defined